## Week 6 Practical Assignment


#### Objective: Understand Spark architecture and perform efficient data processing using transformations, filtering, schema handling, and optimized file formats. Steps: Understand Spark architecture (Driver, Cluster Manager, Executors) and execution modes. Learn Lazy Evaluation and how it optimizes execution using DAG (Lineage Graph). Read data from files (CSV, Parquet) with proper schema handling. Perform filtering and selection of required columns. Modify DataFrames (rename columns, cast data types, add new columns). Apply transformations and actions appropriately. Understand wide transformations and performance concepts (Shuffle, Predicate Pushdown). Work with different file formats (CSV vs Parquet) and their impact on performance. Handle null values and filter datasets efficiently. Build data pipelines (read → transform → filter → write). Save processed data into required formats (CSV/Parquet). Follow best practices for large datasets (avoid collect(), use show()). 

In [0]:
# Dataset
df = spark.table("workspace.default.source")
df.show(5)

+----------+-----------+-------+----------+---------+------+------+--------+----------+-------+
|product_id|   category|  price|  old_name|   status|amount|region|priority|base_price|user_id|
+----------+-----------+-------+----------+---------+------+------+--------+----------+-------+
|         1|Electronics|3215.16|Headphones|Completed|  1928| South|     Low|   3215.16|  190.0|
|         2|Electronics|4466.29|    Bottle|  Pending|   360| North|    High|   4466.29|  130.0|
|         3|     Sports|3029.99|      Book|Cancelled|  5423|  West|    High|   3029.99|   72.0|
|         4|Electronics|4056.68|     Shoes|Cancelled|  3562|  East|  Medium|   4056.68|  196.0|
|         5|Electronics|1716.14|       Bag|Completed|  3040|  East|     Low|   1716.14|   12.0|
+----------+-----------+-------+----------+---------+------+------+--------+----------+-------+
only showing top 5 rows


In [0]:
# View Schema
df.printSchema()

root
 |-- product_id: long (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- old_name: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: long (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- user_id: double (nullable = true)



In [0]:
# Select Required Columns
selected_df= df.select("product_id","category","price")
selected_df.show(5)

+----------+-----------+-------+
|product_id|   category|  price|
+----------+-----------+-------+
|         1|Electronics|3215.16|
|         2|Electronics|4466.29|
|         3|     Sports|3029.99|
|         4|Electronics|4056.68|
|         5|Electronics|1716.14|
+----------+-----------+-------+
only showing top 5 rows


In [0]:
# Filter Electronics
electronics_df = df.filter(df.category == "Electronics")
electronics_df.show(5)

+----------+-----------+-------+----------+---------+------+------+--------+----------+-------+
|product_id|   category|  price|  old_name|   status|amount|region|priority|base_price|user_id|
+----------+-----------+-------+----------+---------+------+------+--------+----------+-------+
|         1|Electronics|3215.16|Headphones|Completed|  1928| South|     Low|   3215.16|  190.0|
|         2|Electronics|4466.29|    Bottle|  Pending|   360| North|    High|   4466.29|  130.0|
|         4|Electronics|4056.68|     Shoes|Cancelled|  3562|  East|  Medium|   4056.68|  196.0|
|         5|Electronics|1716.14|       Bag|Completed|  3040|  East|     Low|   1716.14|   12.0|
|        14|Electronics|2572.16|     Phone|Completed|  1352| South|     Low|   2572.16|   17.0|
+----------+-----------+-------+----------+---------+------+------+--------+----------+-------+
only showing top 5 rows


In [0]:
# Filter using AND
complete_df = df.filter(  (df.status == "Completed") &   (df.amount > 1000) )
complete_df.show(5)

+----------+---------------+-------+----------+---------+------+------+--------+----------+-------+
|product_id|       category|  price|  old_name|   status|amount|region|priority|base_price|user_id|
+----------+---------------+-------+----------+---------+------+------+--------+----------+-------+
|         1|    Electronics|3215.16|Headphones|Completed|  1928| South|     Low|   3215.16|  190.0|
|         5|    Electronics|1716.14|       Bag|Completed|  3040|  East|     Low|   1716.14|   12.0|
|         7|Home Appliances|2907.89|     Table|Completed|  5517| South|  Medium|   2907.89|   60.0|
|        10|       Clothing|2338.19|       Bed|Completed|  5708|  East|    High|   2338.19|    9.0|
|        11|          Books|4034.98|Headphones|Completed|  1828|  East|    High|   4034.98|  102.0|
+----------+---------------+-------+----------+---------+------+------+--------+----------+-------+
only showing top 5 rows


In [0]:
# Cast String to Double
from pyspark.sql.functions import col
cast_df = df.withColumn(
"price",
col("price").cast("double")
)
cast_df.printSchema()

root
 |-- product_id: long (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- old_name: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: long (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- user_id: double (nullable = true)



In [0]:
# Handle Null Values
clean_df = df.filter(
    col("user_id").isNotNull()
)
clean_df.show(5)

+----------+-----------+-------+----------+---------+------+------+--------+----------+-------+
|product_id|   category|  price|  old_name|   status|amount|region|priority|base_price|user_id|
+----------+-----------+-------+----------+---------+------+------+--------+----------+-------+
|         1|Electronics|3215.16|Headphones|Completed|  1928| South|     Low|   3215.16|  190.0|
|         2|Electronics|4466.29|    Bottle|  Pending|   360| North|    High|   4466.29|  130.0|
|         3|     Sports|3029.99|      Book|Cancelled|  5423|  West|    High|   3029.99|   72.0|
|         4|Electronics|4056.68|     Shoes|Cancelled|  3562|  East|  Medium|   4056.68|  196.0|
|         5|Electronics|1716.14|       Bag|Completed|  3040|  East|     Low|   1716.14|   12.0|
+----------+-----------+-------+----------+---------+------+------+--------+----------+-------+
only showing top 5 rows


In [0]:
%sql
CREATE VOLUME workspace.default.week6_volume;

In [0]:
# Save as Parquet
clean_df.write.mode("overwrite").parquet(
"/Volumes/workspace/default/week6_volume/parquet_output"
)

In [0]:
# Read Parquet
parquet_df = spark.read.parquet(
"/Volumes/workspace/default/week6_volume/parquet_output"
)

parquet_df.show(5)

+----------+-----------+-------+----------+---------+------+------+--------+----------+-------+
|product_id|   category|  price|  old_name|   status|amount|region|priority|base_price|user_id|
+----------+-----------+-------+----------+---------+------+------+--------+----------+-------+
|         1|Electronics|3215.16|Headphones|Completed|  1928| South|     Low|   3215.16|  190.0|
|         2|Electronics|4466.29|    Bottle|  Pending|   360| North|    High|   4466.29|  130.0|
|         3|     Sports|3029.99|      Book|Cancelled|  5423|  West|    High|   3029.99|   72.0|
|         4|Electronics|4056.68|     Shoes|Cancelled|  3562|  East|  Medium|   4056.68|  196.0|
|         5|Electronics|1716.14|       Bag|Completed|  3040|  East|     Low|   1716.14|   12.0|
+----------+-----------+-------+----------+---------+------+------+--------+----------+-------+
only showing top 5 rows


In [0]:
# Save as CSV
parquet_df.write.mode("overwrite")\
 .option("header", True) \
 .csv("/Volumes/workspace/default/week6_volume/csv_output")

In [0]:
# Transformations
transform_df = df.select("product_id","category").filter(df.category == "Electronics")


Nothing executes yet because these are transformations.

In [0]:
# Action
transform_df.show()

+----------+-----------+
|product_id|   category|
+----------+-----------+
|         1|Electronics|
|         2|Electronics|
|         4|Electronics|
|         5|Electronics|
|        14|Electronics|
|        20|Electronics|
|        24|Electronics|
|        37|Electronics|
|        40|Electronics|
|        45|Electronics|
|        47|Electronics|
|        51|Electronics|
|        62|Electronics|
|        65|Electronics|
|        80|Electronics|
|        91|Electronics|
|        93|Electronics|
|        94|Electronics|
|        98|Electronics|
|       101|Electronics|
+----------+-----------+
only showing top 20 rows


Execution starts here.

**Spark Architecture**

Driver
• Creates Spark Session
• Builds DAG
• Sends tasks

Cluster Manager
• Allocates CPU and Memory
• Launches Executors

Executors
• Process Data
• Cache Data
• Return Results

**Lazy Evaluation**

Spark uses Lazy Evaluation.

Transformations are not executed immediately.

Spark stores them in a DAG and waits until an Action such as show() or count() is called.

This allows Spark to optimize the execution plan and improve performance.

CSV
- Row-based
- Larger file size
- Slower to read

Parquet
- Columnar format
- Compressed
- Faster queries
• Supports Predicate Pushdown

**Predicate Pushdown**

Predicate Pushdown allows Spark to read only the required rows and columns from a Parquet file instead of scanning the entire dataset.

This reduces memory usage and improves performance.